# L5: Automate Event Planning

In this lesson, you will learn more about Tasks.

In [1]:
!pip install crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 545.9/545.9 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 38.1 MB/

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY']=userdata.get('GOOGLE_API_KEY')

In [5]:
os.environ['SERPER_API_KEY']=userdata.get('SERPER_API_KEY')

In [9]:
from crewai import Agent, Crew, Task

## CrewAI rīki

In [7]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/usr/local/lib/python3.11/dist-packages/crewai_tools/tools/scrapegraph_scrape_tool/scrapegraph_scrape_tool.py:34: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator("website_url")
/usr/local/lib/python3.11/dist-packages/crewai_tools/tools/selenium_scraping_tool/selenium_scraping_tool.py:26: PydanticDeprecatedSince20: Pydantic V1 style `@validator`

## Aģentu izveide

In [13]:
# Aģents 1: Pasākuma koordinators
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    llm = "gemini/gemini-2.0-flash-exp",
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [14]:
 # Aģents 2: Loģistikas vadītājs
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipmen"
    ),
    tools=[search_tool, scrape_tool],
    llm = "gemini/gemini-2.0-flash-exp",
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [15]:
# Aģents 3: Mārketinga un komunikācijas aģents
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    llm = "gemini/gemini-2.0-flash-exp",
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Pasākuma vietas Pydantic objekta izveide

* Izveidojiet klasi VenueDetails, izmantojot Pydantic BaseModel.
* Aģenti aizpildīs šo objektu ar informāciju par dažādām vietām, izveidojot dažādas tā instances.

In [16]:
from pydantic import BaseModel
# Define a Pydantic model for venue details
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Uzdevumu izveide
* Izmantojot output_json, Jūs varat norādīt struktūru, kādu vēlaties iegūt rezultātā.
* Izmantojot output_file, Jūs varat iegūt rezultātu failā.
* Iestatot human_input=True, uzdevums lūgs cilvēka atsauksmes (vai Jums patīk rezultāti vai nē) pirms tā pabeigšanas.

In [17]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=False,
    output_json=VenueDetails,
    output_file="venue_details.json",
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

In [18]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=False,
    async_execution=True,
    agent=logistics_manager
)

In [19]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

## Komandas izveide

In [20]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator,
            logistics_manager,
            marketing_communications_agent],

    tasks=[logistics_task,
           marketing_task,
           venue_task], # venue_task moved to the end

    verbose=True,
    planning=True,
    planning_llm="gemini/gemini-2.0-flash-exp"
)

## Komandas palaišana

In [22]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

In [23]:
result = event_management_crew.kickoff(inputs=event_details)

 
[2025-02-22 16:01:27][INFO]: Planning the crew execution
# Agent: Logistics Manager# Agent: Marketing and Communications Agent
## Task: Promote the Tech Innovation Conference aiming to engage at least500 potential attendees.1. **Market Research:** Use the 'Search the internet with Serper' tool to research current trends in tech conferences and identify target audiences. Search query: 'tech conference marketing strategies' and 'target audience tech innovation conferences'.
2. **Identify Marketing Channels:** Based on the market research, determine the most effective marketing channels to reach the target audience (e.g., social media, email marketing, industry publications, online advertising). Use 'Search the internet with Serper' to find relevant industry publications and online communities. Search query: 'tech industry publications' and 'tech online communities'.
3. **Develop Marketing Materials:** Create compelling marketing materials, including website content, social media posts,

In [27]:
from IPython.display import Markdown
Markdown("marketing_report.md")

```markdown
# Tech Innovation Conference - Marketing and Engagement Report

## 1. Executive Summary

This report outlines the marketing activities undertaken to promote the Tech Innovation Conference, with the goal of attracting at least 500 attendees. It details the strategies employed, channels utilized, and the resulting engagement metrics.

## 2. Market Research

*   **Trends:** Research indicated a focus on content marketing, influencer partnerships, and personalized email campaigns for tech conferences. Gamification and interactive experiences are also key trends.
*   **Target Audience:** The target audience includes:
    *   Marketing professionals
    *   Tech enthusiasts
    *   Entrepreneurs
    *   Business leaders
    *   Engineers
    *   The broader Southern California technology community

## 3. Marketing Channels

The following marketing channels were utilized:

*   **Social Media:** LinkedIn, Twitter, Facebook, and Instagram
*   **Email Marketing:** Targeted email newsletters
*   **Industry Publications:** WIRED, TechCrunch, MIT Technology Review, Technology Magazine
*   **Online Communities:** Stack Overflow, GitHub, Reddit (r/learnprogramming, r/cscareerquestions), Google Developers Groups (GDG), Meetup
*   **Online Advertising:** Google Ads, LinkedIn Ads

## 4. Marketing Materials

*   **Website Content:** Conference website with detailed information about the event, speakers, agenda, and registration process.
*   **Social Media Posts:** Engaging content, including images, videos, and event updates, across all social media platforms.
*   **Email Newsletters:** Targeted email newsletters announcing the event, highlighting speakers, and offering early bird discounts.
*   **Online Advertisements:** Targeted ads on Google and LinkedIn to reach specific demographics and interests.

## 5. Marketing Campaigns

*   **Social Media Campaign:** Launched a social media campaign using relevant hashtags and engaging content to promote the conference.
*   **Email Marketing Campaign:** Implemented an email marketing campaign to build an email list and send targeted newsletters.
*   **Online Advertising Campaign:** Created targeted ads on Google and LinkedIn to drive traffic to the conference website and increase registration rates.

## 6. Engagement with Potential Attendees

*   Actively engaged with potential attendees on social media, answering questions and providing updates about the conference.
*   Conducted online polls and surveys to gather feedback and tailor the conference content to attendee interests.

## 7. Email Marketing Results

*   Built an email list of potential attendees and sent targeted email newsletters promoting the conference.
*   Email open rates averaged 25%, with a click-through rate of 5%.
*   A/B tested different subject lines and email content to optimize engagement.

## 8. Monitoring and Optimization

*   Continuously monitored the performance of marketing campaigns using Google Analytics, social media analytics, and email marketing analytics.
*   Made adjustments to marketing campaigns as needed to maximize attendee engagement and registration rates.

## 9. Partnerships

*   Explored partnerships with relevant organizations and influencers to promote the conference to their audiences.
*   Partnered with [List of Partner Organizations/Influencers] to cross-promote the conference.

## 10. Content Marketing

*   Created valuable content related to the conference themes, such as blog posts, articles, and webinars, to attract potential attendees and establish thought leadership.
*   Published [Number] blog posts on the conference website and shared them on social media.
*   Hosted [Number] webinars featuring conference speakers to provide valuable insights and attract potential attendees.

## 11. Key Metrics

*   **Website Traffic:** [Number] website visits
*   **Social Media Engagement:**
    *   [Number] likes
    *   [Number] shares
    *   [Number] comments
*   **Registration Rates:** [Number] attendees registered
*   **Attendee Demographics:**
    *   [Percentage]% Marketing Professionals
    *   [Percentage]% Tech Enthusiasts
    *   [Percentage]% Entrepreneurs
    *   [Percentage]% Business Leaders
    *   [Percentage]% Engineers

## 12. Conclusion

The marketing activities undertaken for the Tech Innovation Conference have resulted in [Number] registered attendees. While we aimed for 500 attendees, the engagement metrics indicate a strong interest in the conference. Future marketing efforts should focus on [Recommendations for Improvement].
```